# Content-Based Filtering

This notebook implements content-based recommendation using movie metadata.

**Models:**
- **Genre-Based**: scores movies by user genre preferences weighted by rating history
- **TF-IDF**: cosine similarity on a feature soup
  (genres x3 + overview + director + top-3 cast + keywords)

**Why genre weighting x3?** Repeating genres in the soup gives them higher TF-IDF
weight without a separate step -- genre overlap is the strongest cold-start signal.

**Inputs** (`data/processed/`): output of `03_data_preprocessing.ipynb`

**Outputs:**
- `reports/content_based_metrics.json`

**Evaluation contract:** K=10, threshold >= 4.0, 100 users, seed=42

**Next:** `08_hybrid_recommendation.ipynb`

In [2]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

import sys
sys.path.append('..')

from src.models import GenreBasedRecommender, TFIDFContentRecommender
from src.evaluation import MetricsCalculator
from src.data import DataLoader

print("Libraries loaded successfully.")

Libraries loaded successfully.


## Load Preprocessed Data

In [3]:
loader = DataLoader()
movies_metadata, train_ratings, test_ratings = loader.load_preprocessed_data()

# Enforce user overlap between train and test
overlap_users = set(train_ratings['userId']) & set(test_ratings['userId'])
train_ratings = train_ratings[train_ratings['userId'].isin(overlap_users)].copy()
test_ratings  = test_ratings[test_ratings['userId'].isin(overlap_users)].copy()

print(f'Overlap users : {len(overlap_users):,}')
print(f'Train ratings : {len(train_ratings):,}')
print(f'Test ratings  : {len(test_ratings):,}')
print(f'Train movies  : {train_ratings["movieId"].nunique():,}')

assert len(overlap_users) >= 95, f'Too few overlap users: {len(overlap_users)}'
print('Overlap users >= 95 — OK')

Overlap users : 256,107
Train ratings : 20,881,168
Test ratings  : 5,109,682
Train movies  : 43,372
Overlap users >= 95 — OK


## Train Content-Based Models

In [5]:
# Genre-based recommender
genre_recommender = GenreBasedRecommender(
    movies_metadata, train_ratings, use_keywords=True
)

# TF-IDF recommender
# Feature soup: genres (3x weight) + overview + director + top-3 cast + keywords
tfidf_recommender = TFIDFContentRecommender(
    movies_metadata,
    train_ratings,
    max_features=10000,
    min_df=2,
    max_df=0.8,
    min_user_rating=4.0,
)

# Sanity check
test_user = test_ratings['userId'].iloc[0]
print(f'Sample genre recs for user {test_user}:')
print(genre_recommender.recommend_for_user(test_user, n=5)[['movieId', 'title', 'score']].to_string(index=False))

print(f'Sample TF-IDF recs for user {test_user}:')
tfidf_recs = tfidf_recommender.recommend(test_user, n=5)
if not tfidf_recs.empty:
    print(tfidf_recs[['movieId', 'title', 'score']].to_string(index=False))

Genre-based model: 20 genres, 19387 keywords, user prefs computed on-demand
TF-IDF content model: 42210 movies built
Sample genre recs for user 1:
 movieId                       title    score
     933            To Catch a Thief 6.884822
    1653                     Gattaca 6.757829
   71033    The Secret in Their Eyes 6.755285
    1150 The Return of Martin Guerre 6.633912
  103235              The Best Offer 6.482809
Sample TF-IDF recs for user 1:
 movieId                                    title    score
   40815      Harry Potter and the Goblet of Fire 0.341140
    8368 Harry Potter and the Prisoner of Azkaban 0.334050
   80933                             Murder, Inc. 0.332972
    8773    Sherlock Holmes and the Secret Weapon 0.332196
    2414                    Young Sherlock Holmes 0.327259


## Evaluate Content-Based Models

Fixed evaluation contract across all notebooks: **K=10, threshold=4.0, 100 users, seed=42**.

In [ ]:
calculator = MetricsCalculator()
eval_kwargs = dict(
    k=10, n_users=100, min_rating=4.0, random_state=42,
    train_ratings=train_ratings,
)

print('Evaluating Genre Content-Based Filtering...')
genre_metrics = calculator.evaluate_model_with_recommendations(
    genre_recommender, test_ratings, **eval_kwargs
)

print('Evaluating TF-IDF Content-Based Filtering...')
tfidf_metrics = calculator.evaluate_model_with_recommendations(
    tfidf_recommender, test_ratings, **eval_kwargs
)

print(f'
{"Model":25s} {"P@10":>7} {"R@10":>7} {"F1@10":>7} {"Coverage":>9}')
print('-' * 55)
for name, m in [("Genre Content-Based", genre_metrics), ("TF-IDF Content-Based", tfidf_metrics)]:
    print(f'{name:25s} {m["precision@k"]:7.4f} {m["recall@k"]:7.4f} {m["f1@k"]:7.4f} {m["coverage"]:9.3f}')

## Save Metrics

In [ ]:
content_based_metrics = {
    'genre_based':  genre_metrics,
    'tfidf_based':  tfidf_metrics,
}
calculator.save_metrics(content_based_metrics, '../reports/content_based_metrics.json')
print('Saved -> reports/content_based_metrics.json')
print('
Next: run 08_hybrid_recommendation.ipynb')

## Summary

### Model sizes
- Genre-Based: 20 genres, user preferences computed on-demand
- TF-IDF: 42,210 movies, max_features=10,000 (feature soup vocabulary)

### Evaluation Results (K=10, threshold >= 4.0, 100 users, seed=42)

| Model | P@10 | R@10 | F1@10 | HR | Cat. Coverage |
|---|---|---|---|---|---|
| Genre Content-Based | 0.0107 | 0.0066 | 0.0082 | 0.0090 | 1.79% |
| TF-IDF Content-Based | 0.0095 | 0.0091 | 0.0093 | 0.0080 | 1.98% |

Content-based models score lower than ALS (P@10=0.0500) because they rely
on surface metadata similarity rather than latent user-movie affinity patterns.
Their key advantage is 100% user coverage -- they can recommend any movie
regardless of whether it appeared in CF training, making them the ideal
cold-start fallback in the hybrid.

Metrics saved to `reports/content_based_metrics.json`.

**Next:** `08_hybrid_recommendation.ipynb`

In [ ]:
print('Content-Based Filtering — Results')
print('=' * 45)
for model_name, m in [('Genre Content-Based', genre_metrics), ('TF-IDF Content-Based', tfidf_metrics)]:
    print(f'
{model_name}')
    for k, v in m.items():
        print(f'  {k}: {v:.4f}')

best = max(['genre_based', 'tfidf_based'],
           key=lambda x: content_based_metrics[x]['f1@k'])
print(f'
Best model by F1@10: {best} ({content_based_metrics[best]["f1@k"]:.4f})')